# GRAFO DE RELACIONES ENTRE PARTICIPANTES DEL 23F

Instalar librerías requeridas para el análisis

In [37]:
!jupyter nbconvert --to markdown Relaciones_23F_v1_3.ipynb

[NbConvertApp] WARNING | pattern 'Relaciones_23F_v1_3.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--exe

In [ ]:
# Install Libraries
!pip install spacy
!pip install altair
!pip install sentence-transformers
!pip install gensim
!python -m spacy download es_core_news_lg
!pip install networkx pyvis

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
from sentence_transformers import SentenceTransformer
import numpy as np
import requests
from lxml import html
from tqdm import tqdm
import os
import hashlib
import spacy
import gensim
import nltk
import altair as alt
from collections import defaultdict

## Carga de documentos scrapeados

In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv('23f_scrappedDF_v02.csv')

In [ ]:
df.head(1)

## Análisis Estadístico - Dataframe 23F

In [ ]:
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print("\nTipos de dato:")
print(df.dtypes)
print("\nValores nulos:")
print(df.isnull().sum())
print("\nEjemplo de transcript:")
print(df['transcript'][0][:500])

In [ ]:
print(df['transcript'][0])

In [ ]:
df.head(1)

## Limpieza Basica usando spaCy

In [ ]:
# Carga el modelo español grande
nlp = spacy.load("es_core_news_lg")

# Verifica que cargó correctamente
print(nlp.pipe_names)  # ['tok2vec', 'morphologizer', 'parser', 'ner', ...]

Limpieza básica previa antes de pasar el texto a spaCy (Claude lo sugiere así para hacer primero limpieza y luego procesamiento linguistico con spaCy)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)  # control chars
    text = re.sub(r'\n+', ' ', text) # Elimina saltos de línea
    text = re.sub(r'\s+', ' ', text) # Colapsa espacios múltiples
    text = re.sub(r'[^\w\s.,;:áéíóúüñÁÉÍÓÚÜÑ]', '', text) # Conserva puntuación básica y tildes
    return text.strip()

In [ ]:
df['transcript_clean'] = df['transcript'].apply(clean_text)

In [ ]:
df['transcript_clean'][0]

Procesamiento linguistico por batches con spaCy

In [ ]:
# batch_size=32 es un buen punto de partida; auméntalo si tienes RAM suficiente
docs = list(nlp.pipe(df['transcript_clean'], batch_size=32))

# Guarda los objetos Doc en el DataFrame para reutilizarlos
df['spacy_doc'] = docs

In [ ]:
df['spacy_doc'][0]

## Extracción de Features Linguisticos

In [ ]:
def extract_features(doc):
    return {
        "tokens": [token.text for token in doc if not token.is_stop and not token.is_punct],
        "lemmas": [token.lemma_ for token in doc if not token.is_stop and not token.is_punct],
        "entities": [(ent.text, ent.label_) for ent in doc.ents],
        "sentences": [sent.text for sent in doc.sents]
    }

df['nlp_features'] = df['spacy_doc'].apply(extract_features)

In [ ]:
df['nlp_features'][0]

### Named Entities Recognition (NER)

In [ ]:
# Tipos de entidad relevantes para el análisis del 23-F
ENTITY_TYPES = {
    "PER": "Persona",
    "LOC": "Lugar",
    "ORG": "Organización"
}

# Cargar modelo (reutilizar si ya está en memoria desde Fase 0)
nlp = spacy.load("es_core_news_lg")

print(f"Modelo cargado: {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Componentes activos: {nlp.pipe_names}")
print(f"Tipos de entidad a extraer: {list(ENTITY_TYPES.keys())}")

Identificación de Entidades Nombradas en el texto normalizado

In [ ]:
texts = df["transcript_clean"].tolist()
total_docs = len(texts)
BATCH_SIZE = 8

print(f"Procesando {total_docs} documentos con NER (batch_size={BATCH_SIZE})...")

# nlp.pipe() devuelve un generador, tqdm lo envuelve sin romper el batch interno
docs = []

with tqdm(
    nlp.pipe(texts, batch_size=BATCH_SIZE),
    total=total_docs,
    desc="NER Fase 1",
    unit="doc",
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} docs [{elapsed}<{remaining}, {rate_fmt}]"
) as pbar:
    for doc in pbar:
        docs.append(doc)

        # Actualiza el postfix cada BATCH_SIZE documentos para no saturar la UI
        if len(docs) % BATCH_SIZE == 0:
            ents_so_far = sum(
                1 for d in docs for e in d.ents if e.label_ in ENTITY_TYPES
            )
            pbar.set_postfix({
                "entidades": ents_so_far,
                "batch": f"{len(docs) // BATCH_SIZE}/{-(-total_docs // BATCH_SIZE)}"  # ceil division
            })

print(f"\nCompletado. {len(docs)} documentos procesados.")
print(f"Total entidades encontradas: {sum(1 for d in docs for e in d.ents if e.label_ in ENTITY_TYPES)}")

Generacion del DataFrame de Entidades Nombradas

Determinar si es una entidad real o si debe filtrarse, con base en tipología de spaCy

In [ ]:
# Detecta strings con dígitos mezclados con letras (códigos de documento)
CODE_PATTERN = re.compile(r'\d{3,}')  # 3+ dígitos consecutivos = código

def is_valid_entity(ent) -> bool:
    tokens = [token for token in ent]
    text   = ent.text.strip()
    text_l = text.lower()

    # Regla 2: al menos un token debe ser PROPN o NOUN
    if not any(t.pos_ in ("PROPN", "NOUN") for t in tokens):
        return False

    # Regla 3: no puede ser solo ADJ, ADV, VERB o DET
    if all(t.pos_ in ("ADJ", "ADV", "VERB", "DET") for t in tokens):
        return False

    # Regla 4: longitud mínima de caracteres
    if len(text) <= 2:
        return False

    # Regla 5 (nueva): rechazar códigos de documento con dígitos (caso 3)
    if CODE_PATTERN.search(text):
        return False

    # Regla 6 (nueva): rechazar entidades con demasiados tokens, son frases (caso 6)
    if len(tokens) > 5:
        return False

    # Regla 7 (nueva): rechazar si contiene VERB (caso 5: "dale el beso")
    if any(t.pos_ == "VERB" for t in tokens):
        return False

    # Regla 8 (nueva): rechazar si contiene artículos o preposiciones internas
    # (caso 5: "el" en "dale el beso"; caso 6: ruido con stopwords)
    internal_tokens = tokens[1:-1]  # excluye primer y último token
    if any(t.pos_ in ("DET", "ADP") for t in internal_tokens):
        return False

    return True

In [ ]:
# --- Estructura 1: entidades agregadas por documento ---
entity_records = []

for idx, (doc_idx, row) in enumerate(df.iterrows()):
    doc = docs[idx]
    entities = defaultdict(set)

    for ent in doc.ents:
        if ent.label_ in ENTITY_TYPES and is_valid_entity(ent):  # <-- filtro aplicado
            normalized = ent.text.strip().lower()
            entities[ent.label_].add(normalized)

    entity_records.append({
        "doc_id":          row.get("doc_id", doc_idx),
        "persons":         sorted(list(entities.get("PER", []))),
        "locations":       sorted(list(entities.get("LOC", []))),
        "organizations":   sorted(list(entities.get("ORG", []))),
        "n_persons":       len(entities.get("PER", [])),
        "n_locations":     len(entities.get("LOC", [])),
        "n_organizations": len(entities.get("ORG", []))
    })

df_entities = pd.DataFrame(entity_records)

df_entities["total_entities"] = (
    df_entities["n_persons"] +
    df_entities["n_locations"] +
    df_entities["n_organizations"]
)

# --- Estructura 2: registro plano de menciones ---
mention_records = []

for idx, (doc_idx, row) in enumerate(df.iterrows()):
    doc = docs[idx]
    doc_id = row.get("doc_id", doc_idx)

    for ent in doc.ents:
        if ent.label_ in ENTITY_TYPES and is_valid_entity(ent):  # <-- filtro aplicado
            normalized = ent.text.strip().lower()
            ctx_start = max(0, ent.start - 5)
            ctx_end   = min(len(doc), ent.end + 5)

            mention_records.append({
                "doc_id":      doc_id,
                "entity_raw":  ent.text.strip(),
                "entity_norm": normalized,
                "label":       ent.label_,
                "label_desc":  ENTITY_TYPES[ent.label_],
                "context":     doc[ctx_start:ctx_end].text
            })

df_mentions = pd.DataFrame(mention_records)

print(f"df_entities shape: {df_entities.shape}")
print(f"df_mentions shape: {df_mentions.shape}")
print(f"\nTotal menciones por tipo:\n{df_mentions['label'].value_counts()}")

In [ ]:
# Entidades a inyectar manualmente por documento
MANUAL_INJECTIONS = [
    {"doc_id": 61, "entity_raw": "Consejo de Guerra", "entity_norm": "consejo de guerra", "label": "ORG"},
    # Añade más filas aquí si aparecen casos similares en otros docs
]

for inj in MANUAL_INJECTIONS:
    inj["label_desc"]     = ENTITY_TYPES.get(inj["label"], inj["label"])
    inj["context"]        = ""
    inj["label_resolved"] = inj["label"]

df_injections = pd.DataFrame(MANUAL_INJECTIONS)
df_mentions   = pd.concat([df_mentions, df_injections], ignore_index=True)

print(f"Inyecciones aplicadas: {len(MANUAL_INJECTIONS)}")
print(df_mentions[df_mentions["doc_id"] == 61])

## Revisión de calidad de las entidades encontradas

In [ ]:
# Entidades más frecuentes por tipo
print("=" * 50)
print("TOP 10 ENTIDADES POR TIPO")
print("=" * 50)

for label, desc in ENTITY_TYPES.items():
    top = (
        df_mentions[df_mentions["label"] == label]["entity_norm"]
        .value_counts()
        .head(10)
    )
    print(f"\n{desc} ({label}):")
    print(top.to_string())

# Documentos con mayor densidad de entidades
print("\n" + "=" * 50)
print("DOCUMENTOS CON MÁS ENTIDADES")
print("=" * 50)

print(df_entities[["doc_id", "n_persons", "n_locations", "n_organizations", "total_entities"]]
      .sort_values("total_entities", ascending=False)
      .head(10)
      .to_string(index=False))

# Muestra de menciones con contexto (útil para revisar falsos positivos)
print("\n" + "=" * 50)
print("MUESTRA DE MENCIONES CON CONTEXTO")
print("=" * 50)
print(df_mentions.sample(min(10, len(df_mentions)))[
    ["doc_id", "label_desc", "entity_norm", "context"]
].to_string(index=False))

Corrección de Conflictos Identificados entre Entidades

In [ ]:
# Contar menciones por (entity_norm, label)
vote_counts = (
    df_mentions
    .groupby(["entity_norm", "label"])
    .size()
    .reset_index(name="count")
)

# Para cada entidad, quedarse con el label de mayor frecuencia
winning_label = (
    vote_counts
    .sort_values("count", ascending=False)
    .drop_duplicates(subset="entity_norm", keep="first")
    .set_index("entity_norm")["label"]
)

# Identificar entidades con conflicto (aparecen en más de un tipo)
conflict_entities = (
    vote_counts
    .groupby("entity_norm")["label"]
    .nunique()
    .loc[lambda x: x > 1]
    .index.tolist()
)

print(f"Entidades con conflicto de tipo: {len(conflict_entities)}")
print(f"\nDetalle de conflictos:")
print(
    vote_counts[vote_counts["entity_norm"].isin(conflict_entities)]
    .sort_values(["entity_norm", "count"], ascending=[True, False])
    .to_string(index=False)
)

Agregar Prioridad en Reconocimiento de Entidades

In [ ]:
# Prioridad: 1 = máxima prioridad
LABEL_PRIORITY = {"PER": 1, "ORG": 2, "LOC": 3}

def resolve_label(entity_norm, vote_counts_df, priority_map):
    """
    Resuelve conflictos combinando frecuencia y prioridad de dominio.
    Si el label mayoritario y el de mayor prioridad difieren,
    el de mayor prioridad gana solo si su frecuencia >= 30% del total.
    """
    rows = vote_counts_df[vote_counts_df["entity_norm"] == entity_norm].copy()

    if len(rows) == 1:
        return rows.iloc[0]["label"]

    total = rows["count"].sum()
    rows["pct"] = rows["count"] / total
    rows["priority"] = rows["label"].map(priority_map)

    # Label de mayor prioridad con al menos 30% de las menciones
    high_priority = rows[rows["pct"] >= 0.30].sort_values("priority")

    if not high_priority.empty:
        return high_priority.iloc[0]["label"]

    # Si ninguno alcanza el 30%, gana el mayoritario
    return rows.sort_values("count", ascending=False).iloc[0]["label"]


# Construir mapa definitivo entity_norm -> label correcto
resolved_labels = {
    entity: resolve_label(entity, vote_counts, LABEL_PRIORITY)
    for entity in conflict_entities
}

# Completar con entidades sin conflicto
for entity, label in winning_label.items():
    if entity not in resolved_labels:
        resolved_labels[entity] = label

print("Resolución de conflictos:")
for entity in conflict_entities:
    rows = vote_counts[vote_counts["entity_norm"] == entity]
    final = resolved_labels[entity]
    print(f"  '{entity}': {dict(zip(rows['label'], rows['count']))} -> {final}")

Agregar Whitelist de Entidades Conocidas con base en el Contexto

In [ ]:
# Entidades clave del 23-F con su tipo correcto
KNOWN_ENTITIES_23F = {
    # Personas
    "rey":                 "PER",
    "tejero":              "PER",
    "milans del bosch":    "PER",
    "armada":              "PER",
    "suárez":              "PER",
    "gutiérrez mellado":   "PER",
    "calvo sotelo":        "PER",
    "aramburu":            "PER",
    # Lugares
    "valencia":            "LOC",
    "madrid":              "LOC",
    "españa":              "LOC",
    "málaga":              "LOC",
    "barcelona":           "LOC",
    "bilbao":              "LOC",
    "burgos":              "LOC",
    "canarias":            "LOC",
    "ceuta":               "LOC",
    "getafe":              "LOC",
    "oviedo":              "LOC",
    "pamplona":            "LOC",
    "sevilla":             "LOC",
    "cuartel general":     "LOC",
    # Organizaciones
    "guardia civil":       "ORG",
    "psoe":                "ORG",
    "ucd":                 "ORG",
    "gobierno":            "ORG",
    "congreso":            "ORG",
    "congreso de los diputados":     "ORG",
    "brigada":             "ORG",
    "colegio de abogados":           "ORG",
    "inf":                 "ORG",
    "ministerio de defensa":         "ORG",
    "ministerio del interior":       "ORG",
    "parlamento":          "ORG",
    "tribunal":          "ORG",
    "Consejo de Guerra":   "ORG"
}

# La whitelist sobreescribe cualquier resultado previo
resolved_labels.update(KNOWN_ENTITIES_23F)

print(f"Whitelist aplicada: {len(KNOWN_ENTITIES_23F)} entidades fijadas manualmente.")

Corrección de Entidades con Base en Estrategias y Whitelist

In [ ]:
# Corregir el label en df_mentions
df_mentions["label_resolved"] = df_mentions["entity_norm"].map(resolved_labels)

# Verificar que no haya NaN (entidades sin resolución)
missing = df_mentions[df_mentions["label_resolved"].isna()]["entity_norm"].unique()
if len(missing) > 0:
    print(f"Advertencia: {len(missing)} entidades sin label resuelto. Rellenando con label original.")
    df_mentions["label_resolved"] = df_mentions["label_resolved"].fillna(df_mentions["label"])

# Reconstruir df_entities con los labels corregidos
entity_records_clean = []

for doc_id, group in df_mentions.groupby("doc_id"):
    entities = defaultdict(set)
    for _, row in group.iterrows():
        entities[row["label_resolved"]].add(row["entity_norm"])

    entity_records_clean.append({
        "doc_id":          doc_id,
        "persons":         sorted(list(entities.get("PER", []))),
        "locations":       sorted(list(entities.get("LOC", []))),
        "organizations":   sorted(list(entities.get("ORG", []))),
        "n_persons":       len(entities.get("PER", [])),
        "n_locations":     len(entities.get("LOC", [])),
        "n_organizations": len(entities.get("ORG", [])),
    })

df_entities_clean = pd.DataFrame(entity_records_clean)
df_entities_clean["total_entities"] = (
    df_entities_clean["n_persons"] +
    df_entities_clean["n_locations"] +
    df_entities_clean["n_organizations"]
)

print(f"df_entities_clean shape: {df_entities_clean.shape}")
print(f"\nTotal menciones por tipo (después de corrección):")
print(df_mentions["label_resolved"].value_counts())

In [ ]:
df_entities_clean.head(5)

Merge de Entidades Corregidas con el DF Original

In [ ]:
# Añadir doc_id al DataFrame original si no existe
if "doc_id" not in df.columns:
    df["doc_id"] = df.index

df = df.merge(
    df_entities_clean[[
        "doc_id", "persons", "locations", "organizations",
        "n_persons", "n_locations", "n_organizations", "total_entities"
    ]],
    on="doc_id",
    how="left"
)

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print(f"Nulos en persons: {df['persons'].isna().sum()}")  # debe ser 0

In [ ]:
# Ver qué IDs no matchearon
ids_df      = set(df["doc_id"])
ids_clean   = set(df_entities_clean["doc_id"])
print("En df pero no en df_entities_clean:", ids_df - ids_clean)

In [ ]:
df.loc[df['doc_id'] == 61]

In [ ]:
df_entities_clean.loc[[60]]

In [ ]:
df['transcript_clean'][61]

In [ ]:
# Paso 1: ¿El doc tiene menciones en df_mentions?
print(df_mentions[df_mentions["doc_id"] == 61])

# Paso 2: ¿Qué extrajo spaCy antes del filtro?
doc_61 = docs[61]
print("Entidades crudas de spaCy:")
for ent in doc_61.ents:
    print(f"  '{ent.text}' | label={ent.label_} | valid={is_valid_entity(ent)}")

# Paso 3: ¿Qué dice el transcript?
print("\nTranscript limpio:")
print(df["transcript_clean"].iloc[61][:500])

Exportar Outputs

In [ ]:
# Guardar para Fase 2
df_entities_clean.to_csv("fase1_entities_by_doc.csv", index=False)
df_mentions.to_csv("fase1_mentions.csv", index=False)
df_entities_clean.to_markdown("entities_clean.md", tablefmt="grid")
df_mentions.to_markdown("mentions.md", tablefmt="grid")

print("Outputs de Fase 1 guardados:")
print("  - fase1_entities_by_doc.csv  (entidades por documento)")
print("  - fase1_mentions.csv         (registro plano de menciones)")
print(f"\nDataFrame principal df ahora tiene {df.shape[1]} columnas.")
print(f"Columnas nuevas: persons, locations, organizations, n_*, total_entities")

## Grafo de Coocurrencia entre Entidades

### Construcción de Pares de Coocurrencia

In [ ]:
from itertools import combinations
from collections import defaultdict
import networkx as nx

# Mínimo de documentos donde deben co-ocurrir dos entidades para crear arco
MIN_COOC = 10

edge_weights = defaultdict(int)

for _, row in df_entities_clean.iterrows():
    # Recopilar todas las entidades del documento con su tipo
    all_entities = (
        [(e, "PER") for e in row["persons"]]
        + [(e, "ORG") for e in row["organizations"]]
        + [(e, "LOC") for e in row["locations"]]
    )

    # Generar todos los pares únicos dentro del mismo documento
    for (e1, t1), (e2, t2) in combinations(all_entities, 2):
        edge_key = tuple(sorted([e1, e2]))  # arco no dirigido
        edge_weights[edge_key] += 1

print(f"Pares únicos encontrados: {len(edge_weights)}")
print(f"Pares con co-ocurrencia >= {MIN_COOC}: {sum(1 for v in edge_weights.values() if v >= MIN_COOC)}")

### Generar Grafo de Entidades

In [ ]:
# Mapa entidad -> tipo (desde df_mentions ya resuelto)
entity_type_map = (
    df_mentions
    .drop_duplicates(subset="entity_norm")
    .set_index("entity_norm")["label_resolved"]
    .to_dict()
)

# Colores por tipo para visualización
TYPE_COLORS = {"PER": "#4C9BE8", "ORG": "#E8874C", "LOC": "#6ABF69", "UNK": "#AAAAAA"}

G = nx.Graph()

for (e1, e2), weight in edge_weights.items():
    if weight >= MIN_COOC:
        G.add_edge(e1, e2, weight=weight)

# Añadir atributos a los nodos
for node in G.nodes():
    etype = entity_type_map.get(node, "UNK")
    G.nodes[node]["type"]  = etype
    G.nodes[node]["color"] = TYPE_COLORS[etype]

print(f"Nodos: {G.number_of_nodes()}")
print(f"Arcos: {G.number_of_edges()}")
print(f"Componentes conectados: {nx.number_connected_components(G)}")

### Evaluar Metricas de Centralidad

In [ ]:
# Centralidades principales
degree_cent     = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G, weight="weight")
weighted_degree  = dict(G.degree(weight="weight"))

df_centrality = pd.DataFrame({
    "entity":      list(G.nodes()),
    "type":        [G.nodes[n]["type"] for n in G.nodes()],
    "degree":      [G.degree(n) for n in G.nodes()],
    "weighted_deg":[weighted_degree[n] for n in G.nodes()],
    "degree_cent": [degree_cent[n] for n in G.nodes()],
    "betweenness": [betweenness_cent[n] for n in G.nodes()],
}).sort_values("betweenness", ascending=False)

print("Top 15 entidades por betweenness centrality:")
print(df_centrality.head(15).to_string(index=False))

### Visualización del Grafo de Entidades (Matplotlib)

In [ ]:
# Filtrar solo nodos con degree suficiente para no saturar el grafo
MIN_DEGREE = 11
subgraph_nodes = [n for n, d in G.degree() if d >= MIN_DEGREE]
G_sub = G.subgraph(subgraph_nodes)

plt.figure(figsize=(18, 14))

pos = nx.spring_layout(G_sub, weight="weight", seed=42, k=0.6)

node_colors = [G_sub.nodes[n]["color"] for n in G_sub.nodes()]
node_sizes  = [300 + G_sub.degree(n) * 40 for n in G_sub.nodes()]
edge_widths = [G_sub[u][v]["weight"] * 0.3 for u, v in G_sub.edges()]

nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors, node_size=node_sizes, alpha=0.85)
nx.draw_networkx_edges(G_sub, pos, width=edge_widths, alpha=0.4, edge_color="#888888")
nx.draw_networkx_labels(G_sub, pos, font_size=7, font_color="black")

# Leyenda
from matplotlib.patches import Patch
legend = [Patch(color=v, label=k) for k, v in TYPE_COLORS.items() if k != "UNK"]
plt.legend(handles=legend, loc="upper left", fontsize=10)

plt.title("Grafo de Co-ocurrencia de Entidades — 23F", fontsize=14, pad=20)
plt.axis("off")
plt.tight_layout()
plt.savefig("fase2_grafo_entidades.png", dpi=150, bbox_inches="tight")
plt.show()

### Grafo de Entidades Interactivo (PyVis)

In [ ]:
from pyvis.network import Network

net = Network(height="750px", width="100%", bgcolor="#1a1a2e", font_color="white")
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=150)

for node in G_sub.nodes(data=True):
    name, attrs = node
    size = 10 + G_sub.degree(name) * 2
    net.add_node(name, label=name, color=attrs["color"], size=size,
                 title=f"{attrs['type']}: {name}\nDegree: {G_sub.degree(name)}")

for u, v, data in G_sub.edges(data=True):
    net.add_edge(u, v, width=data["weight"] * 0.3,
                 title=f"Co-ocurrencias: {data['weight']}")

net.save_graph("fase2_grafo_interactivo.html")
print("Grafo interactivo guardado: fase2_grafo_interactivo.html")